# Signals of Prestige in Film Discourse: A Transformer-Based Approach to Predicting Best Picture Winners

### By: Ed Hou, Si Qin Huang, Alejandro Mendez

---

Click here to [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-huang23/nlp-oscar-predictor/blob/model_dev/oscar_predictor.ipynb)


In [14]:
!pip install kagglehub

In [15]:
import kagglehub
import os
import glob
import json
import pandas as pd
import numpy as np
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, AutoModel, AutoTokenizer
from typing import Optional

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import scipy.sparse as sp

In [16]:
# Mount drive (If manually uploaded data, mount drive)
# from google.colab import drive
# drive.mount('/content/drive')

# Clone only the data folder from the repository (if not already present)
if not os.path.exists("data"):
    os.system("rm -rf /tmp/nlp-repo")
    ret = os.system("git clone --depth=1 https://github.com/s-huang23/nlp-oscar-predictor.git /tmp/nlp-repo")
    if ret == 0:
        os.system("cp -r /tmp/nlp-repo/data .")
        print("Data cloned successfully.")
    else:
        print("ERROR: git clone failed. Check that the repo is public and the URL is correct.")

print("data/ exists:", os.path.exists("data"))
print("data/raw/ exists:", os.path.exists("data/raw"))

Data cloned successfully.
data/ exists: True
data/raw/ exists: True


## Dataset (NEEDS TO BE IMPLEMENTED)

Load iMDb data from kaggle and web-scraped reviews from Metacritic

#### Metacritic data

In [17]:
# Load metacritic dataset
# metacritic_df = pd.read_csv("/content/drive/MyDrive/metacritic_reviews.csv") # If data is uploaded to drive
metacritic_df = pd.read_csv("data/raw/metacritic_reviews.csv")
print(metacritic_df.shape)
metacritic_df.head()

(6643, 13)


,ceremony_year,release_year,film_title,winner,metacritic_slug,metacritic_url,critic_review_page,review_date,critic_score,publication,author,quote,full_review_url
0,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,100,The Hollywood Reporter,Kirk Honeycutt,"A fully believable, flesh-and-blood (albeit no...",http://www.hollywoodreporter.com/hr/film-revie...
1,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,100,Empire,Chris Hewitt (1),"It's been twelve years since ""Titanic,"" but th...",http://www.empireonline.com/reviews/reviewcomp...
2,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,90,Variety,Todd McCarthy,"Avatar is all-enveloping and transporting, wit...",http://www.variety.com/review/VE1117941773.htm...
3,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,100,Chicago Sun-Times,Roger Ebert,"Once again, [Cameron] has silenced the doubter...",http://rogerebert.suntimes.com/apps/pbcs.dll/a...
4,2010,2009,Avatar,0,avatar,https://www.metacritic.com/movie/avatar/,https://www.metacritic.com/movie/avatar/critic...,NaN,75,Chicago Tribune,Michael Phillips,The first 90 minutes of Avatar are pretty terr...,http://featuresblogs.chicagotribune.com/talkin...


#### Kaggle iMDb data

In [18]:
# Load nominees dataset
# nominees_df = pd.read_csv("/content/drive/MyDrive/nominees.csv") # If data is uploaded to drive
nominees_df   = pd.read_csv("data/nominees.csv")
print(nominees_df.shape)
nominees_df.head()

(136, 5)


,ceremony_year,film_title,release_year,winner,metacritic_slug
0,2010,Avatar,2009,0,avatar
1,2010,The Blind Side,2009,0,the-blind-side
2,2010,District 9,2009,0,district-9
3,2010,An Education,2009,0,an-education
4,2010,The Hurt Locker,2009,1,the-hurt-locker


In [19]:
# Download latest version of data from kaggle
path = kagglehub.dataset_download("ebiswas/imdb-review-dataset")

print("Path to dataset files:", path)

100%|██████████| 2.69G/2.69G [00:20<00:00, 138MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ebiswas/imdb-review-dataset/versions/1


In [ ]:
# Keep Metacritic / nominee titles as the canonical film titles.
# Kaggle IMDb titles sometimes include alternate punctuation, accents, or IMDb roman-numeral suffixes.
# These variants are mapped back to the Metacritic title before merging.
imdb_to_metacritic_title = {
    "Extremely Loud & Incredibly Close": "Extremely Loud and Incredibly Close",
    "Les Misérables": "Les Miserables",
    "Once Upon a Time... In Hollywood": "Once Upon a Time ... in Hollywood",
    "Hell or High Water (II)": "Hell or High Water",
    "Boyhood (I)": "Boyhood",
    "Arrival (II)": "Arrival",
    "Get Out (I)": "Get Out",
    "Moonlight (I)": "Moonlight",
    "Room (I)": "Room",
    "Spotlight (I)": "Spotlight",
    "The Artist (I)": "The Artist",
    "Vice (I)": "Vice"
}

# nominees_df and metacritic_df already use canonical Metacritic titles.
# Apply imdb_to_metacritic_title to reviews_df["movie"] after extracting IMDb movie titles.


In [ ]:
# print(os.listdir(path))

In [ ]:
# Combine all parts of json
all_files = glob.glob(os.path.join(path, "part-*.json"))

records = []
for f in all_files:
    with open(f) as fh:
        records.extend(json.load(fh))

# Convert to dataframe
reviews_df = pd.DataFrame(records)
print(reviews_df.shape)
reviews_df.head()

(5571499, 9)


,review_id,reviewer,movie,rating,review_summary,review_date,spoiler_tag,review_detail,helpful
0,rw1133942,OriginalMovieBuff21,Kill Bill: Vol. 2 (2004),8,Good follow up that answers all the questions,24 July 2005,0,"After seeing Tarantino's Kill Bill Vol: 1, I g...","[0, 1]"
1,rw1133943,sentra14,Journey to the Unknown (1968â€“ ),None,Excellent series,24 July 2005,0,"I have the entire series on video, taped mostl...","[11, 11]"
2,rw1133946,GreenwheelFan2002,The Island (2005),9,"Not just about action, but about survival...",24 July 2005,0,Once again the critics prove themselves as mor...,"[2, 5]"
3,rw1133948,itsascreambaby,Win a Date with Tad Hamilton! (2004),3,Falls under the category: seen it a million ti...,24 July 2005,0,This IS a film that has been done too many tim...,"[2, 3]"
4,rw1133949,OriginalMovieBuff21,Saturday Night Live: The Best of Chris Farley ...,10,"Before Tommy Boy and Black Sheep, there was Sa...",24 July 2005,0,Chris Farley is one of my favorite comedians a...,"[4, 4]"


In [ ]:
# Extract title and year from "Movie Title (YEAR)" format
reviews_df["release_year"] = (
    reviews_df["movie"]
    .str.extract(r'\((\d{4})')   # grab the first 4-digit year after '('
    .astype(float)
    .astype("Int64")
)

reviews_df["movie"] = (
    reviews_df["movie"]
    .str.replace(r'\s*\(\d{4}[^)]*\)', '', regex=True)  # remove anything from (YEAR...  to )
    .str.strip()
)


# Convert Kaggle IMDb title variants to canonical Metacritic / nominee titles.
reviews_df["movie"] = reviews_df["movie"].replace(imdb_to_metacritic_title)

print(reviews_df[["movie", "release_year"]].head(10))

                                           movie  release_year
0                              Kill Bill: Vol. 2          2004
1                         Journey to the Unknown          1968
2                                     The Island          2005
3                  Win a Date with Tad Hamilton!          2004
4  Saturday Night Live: The Best of Chris Farley          2000
5                                    Outlaw Star          1998
6                                    The Aviator          2004
7      Star Wars: Episode I - The Phantom Menace          1999
8                          The Amityville Horror          2005
9                                  Flying Tigers          1942


In [ ]:
# Filter and merge using both columns
filtered_reviews = reviews_df[
    reviews_df["movie"].isin(nominees_df["film_title"]) &
    reviews_df["release_year"].isin(nominees_df["release_year"])
].copy()

print(f"Total reviews matched: {len(filtered_reviews)}")
print(f"Unique films matched : {filtered_reviews['movie'].nunique()} / {len(nominees_df)}")

imdb_df = filtered_reviews.merge(
    nominees_df[["film_title", "release_year", "ceremony_year", "winner"]],
    left_on=["movie", "release_year"],
    right_on=["film_title", "release_year"],
    how="left"
)

# Drop values with NAs that don't match up
imdb_df = imdb_df.dropna(subset=["winner"])
# Fix winner and ceremony_year column formatting
imdb_df["winner"] = imdb_df["winner"].astype(int)
imdb_df["ceremony_year"] = imdb_df["ceremony_year"].astype(int)

print(imdb_df.shape)
imdb_df.head()

Total reviews matched: 84642
Unique films matched : 87 / 96
(84107, 13)


,review_id,reviewer,movie,rating,review_summary,review_date,spoiler_tag,review_detail,helpful,release_year,film_title,ceremony_year,winner
0,rw2531251,galvanekps,The Help,8,You just never know...,12 December 2011,0,I have to state up front I went into watching ...,"[0, 5]",2011,The Help,2012,0
1,rw2531254,remedy305,Hugo,8,1920's Paris comes to life,12 December 2011,0,"This is a wonderful story about a boy, Hugo, w...","[4, 6]",2011,Hugo,2012,0
2,rw2531322,gnguyen102,Midnight in Paris,9,Well-written script and outstanding acting,12 December 2011,0,I must admit that I have been a fan of Woody A...,"[0, 1]",2011,Midnight in Paris,2012,0
3,rw2531410,mmobini,Hugo,7,A huge potential for an exceptional film.,12 December 2011,0,A resplendently sweeping opening shot that pro...,"[2, 5]",2011,Hugo,2012,0
4,rw2531412,Abigail_fox,Hugo,1,Couldn't wait for it to end - so dull!,12 December 2011,0,I saw this film yesterday. I didn't see it in ...,"[88, 176]",2011,Hugo,2012,0


In [ ]:
# Drop unneeded columns
imdb_df = imdb_df.drop(columns=["movie", "review_summary", "spoiler_tag", "helpful"])

# Strip disambiguation suffixes like (I), (II) from film titles
for df in [imdb_df, nominees_df, metacritic_df]:
    df["film_title"] = df["film_title"].str.replace(r'\s*\([IV]+\)$', '', regex=True).str.strip()

print(imdb_df.columns.tolist())
print(imdb_df["winner"].value_counts())
imdb_df.head()

['review_id', 'reviewer', 'rating', 'review_date', 'review_detail', 'release_year', 'film_title', 'ceremony_year', 'winner']
winner
0    73552
1    10555
Name: count, dtype: int64


,review_id,reviewer,rating,review_date,review_detail,release_year,film_title,ceremony_year,winner
0,rw2531251,galvanekps,8,12 December 2011,I have to state up front I went into watching ...,2011,The Help,2012,0
1,rw2531254,remedy305,8,12 December 2011,"This is a wonderful story about a boy, Hugo, w...",2011,Hugo,2012,0
2,rw2531322,gnguyen102,9,12 December 2011,I must admit that I have been a fan of Woody A...,2011,Midnight in Paris,2012,0
3,rw2531410,mmobini,7,12 December 2011,A resplendently sweeping opening shot that pro...,2011,Hugo,2012,0
4,rw2531412,Abigail_fox,1,12 December 2011,I saw this film yesterday. I didn't see it in ...,2011,Hugo,2012,0


## Pre-processing

Both IMDb and Metacritic reviews are processed with the same core leakage-control and cleanup steps.

Implemented steps:
  - Filter to Oscar ceremony years 2012-2020
  - Filter each review to before the ceremony date: `review_date < ceremony_date`
  - Remove rows with invalid dates, short or empty text, and duplicate review text
  - Clean review text into `clean_text` while preserving emoji and sentiment-bearing punctuation
  - Add `film_review_count` and `film_review_weight` so films with more reviews do not dominate downstream aggregation

The ceremony date is excluded because review timestamps are date-only; excluding Oscar day avoids accidentally including reviews written after the winner was announced.


In [ ]:
# Preprocess IMDb and Metacritic reviews.
# Core window: review_date < ceremony_date.
import re

START_YEAR = 2012
END_YEAR = 2020
MIN_TEXT_CHARS = 20

# windows_path = "/content/drive/MyDrive/oscar_windows.csv"
oscar_windows = pd.read_csv("data/oscar_windows.csv", parse_dates=["nomination_date", "ceremony_date"])


def clean_text(value):
    """Normalize review text while preserving the words used by reviewers."""
    text = "" if pd.isna(value) else str(value)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# Sanity check: emoji can carry sentiment, so preprocessing should keep them.
assert clean_text("Loved it 😭!") == "Loved it 😭!"


def summarize_reviews(label, df):
    film_count = df[["ceremony_year", "film_title"]].drop_duplicates().shape[0]
    years = sorted(df["ceremony_year"].dropna().astype(int).unique().tolist()) if len(df) else []
    print(f"{label:<36} rows={len(df):>6} films={film_count:>3} years={years}")


def preprocess_reviews(df, source_name, text_col):
    """Apply project preprocessing to one review source."""
    print(f"\n{source_name.upper()} preprocessing")
    summarize_reviews("raw", df)

    working = df[
        (df["ceremony_year"] >= START_YEAR) &
        (df["ceremony_year"] <= END_YEAR)
    ].copy()
    summarize_reviews("after ceremony-year filter", working)

    working = working.merge(oscar_windows, on="ceremony_year", how="left", validate="many_to_one")
    working["review_date_parsed"] = pd.to_datetime(working["review_date"], errors="coerce")

    invalid_dates = working["review_date_parsed"].isna().sum()
    if invalid_dates:
        print(f"dropped invalid review dates: {invalid_dates}")

    in_window = (
        working["review_date_parsed"].notna()
        & (working["review_date_parsed"] < working["ceremony_date"])
    )
    working = working[in_window].copy()
    summarize_reviews("after ceremony-date filter", working)

    working["clean_text"] = working[text_col].map(clean_text)
    before = len(working)
    working = working[working["clean_text"].str.len() >= MIN_TEXT_CHARS].copy()
    print(f"dropped short/empty reviews: {before - len(working)}")

    dedupe_cols = ["ceremony_year", "film_title", "clean_text"]
    if source_name == "imdb" and "reviewer" in working.columns:
        dedupe_cols.insert(2, "reviewer")
    if source_name == "metacritic" and "publication" in working.columns:
        dedupe_cols.insert(2, "publication")

    before = len(working)
    working = working.drop_duplicates(subset=dedupe_cols).copy()
    print(f"dropped duplicate reviews: {before - len(working)}")

    working["film_review_count"] = working.groupby(
        ["ceremony_year", "film_title"]
    )["clean_text"].transform("size")
    working["film_review_weight"] = 1.0 / working["film_review_count"]

    summarize_reviews("final", working)
    return working


imdb_df = preprocess_reviews(imdb_df, "imdb", "review_detail")
metacritic_df = preprocess_reviews(metacritic_df, "metacritic", "quote")


## Model

**Pipeline**: <br>
BERT  â†’ [CLS] per review  â†’ WeightedAggregator â†’ review film vector <br>
BERTweet â†’ [CLS] per tweet â†’ WeightedAggregator â†’ tweet film vector <br>
Concatenate â†’ Classification Head â†’ P(win)

**Weighted Aggregator**: aggregates review embeddings from BERT and BERTweet into a single film vector

### BERT (On reviews + tweets)

In [ ]:
class OscarPredictorBERT(nn.Module):
    """
    Baseline model: single shared BERT encoder for both reviews and tweets.
    Uses mean pooling instead of learned aggregation.
    """
    def __init__(
        self,
        model_name="bert-base-uncased",
        hidden_dim=768,
        dropout=0.3
    ):
        super(OscarPredictorBERT, self).__init__()

        # Single shared encoder for both reviews and tweets
        self.encoder = AutoModel.from_pretrained(model_name)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def encode(self, input_ids, attention_mask):
        """
        Encode a batch of texts and return mean-pooled CLS representations.

        input_ids     : (batch_size, num_texts, seq_len)
        attention_mask: (batch_size, num_texts, seq_len)
        returns       : (batch_size, hidden_dim)
        """
        batch_size, num_texts, seq_len = input_ids.shape

        input_ids = input_ids.reshape(-1, seq_len)
        attention_mask = attention_mask.reshape(-1, seq_len)

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:, 0, :]
        cls = cls.view(batch_size, num_texts, -1)
        return cls.mean(dim=1)

    def forward(
        self,
        review_input_ids,
        review_attention_mask,
        tweet_input_ids,
        tweet_attention_mask
    ):
        review_agg = self.encode(review_input_ids, review_attention_mask)
        tweet_agg  = self.encode(tweet_input_ids,  tweet_attention_mask)

        film_vector = torch.cat([review_agg, tweet_agg], dim=1)

        return self.classifier(film_vector)

### TF-IDF + Logistic Regression

In [ ]:
class OscarPredictorTFIDF:
    """
    Baseline model: TF-IDF + Logistic Regression.
    Reviews and tweets are each vectorized separately then concatenated.
    """

    def __init__(self, max_features=10000, C=1.0):
        # Separate vectorizers so each modality gets its own vocab
        self.review_vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),    # unigrams + bigrams
            sublinear_tf=True,     # apply log normalization to term freq
            strip_accents="unicode",
            min_df=2,              # ignore terms appearing in fewer than 2 films
        )
        self.tweet_vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=(1, 2),
            sublinear_tf=True,
            strip_accents="unicode",
            min_df=2,
        )
        self.classifier = LogisticRegression(
            C=C,                   # inverse regularization strength
            max_iter=1000,
            class_weight="balanced",  # handles 1 winner vs many nominees imbalance
        )

    def _prepare_features(self, reviews, tweets, fit=False):
        """
        Vectorize reviews and tweets and concatenate into one feature matrix.

        reviews : list of str â€” one string per film (all reviews concatenated)
        tweets  : list of str â€” one string per film (all tweets concatenated)
        returns : sparse matrix (num_films, 2 * max_features)
        """
        if fit:
            review_features = self.review_vectorizer.fit_transform(reviews)
            tweet_features  = self.tweet_vectorizer.fit_transform(tweets)
        else:
            review_features = self.review_vectorizer.transform(reviews)
            tweet_features  = self.tweet_vectorizer.transform(tweets)

        # Horizontally stack review and tweet features
        return sp.hstack([review_features, tweet_features])

    def fit(self, reviews, tweets, labels):
        """
        Train the model.

        reviews : list of str, one per film
        tweets  : list of str, one per film
        labels  : list of int, 1 = winner, 0 = nominee
        """
        X = self._prepare_features(reviews, tweets, fit=True)
        self.classifier.fit(X, labels)

    def predict_proba(self, reviews, tweets):
        """
        Returns P(win) for each film.

        returns: np.array of shape (num_films,)
        """
        X = self._prepare_features(reviews, tweets, fit=False)
        return self.classifier.predict_proba(X)[:, 1]

    def predict_winner(self, reviews, tweets):
        """
        Returns the index of the predicted winner within the provided films
        (intended to be called with one year's nominees at a time).
        """
        probs = self.predict_proba(reviews, tweets)
        return np.argmax(probs)

## Evaluation (NEEDS TO BE IMPLEMENTED)
Methods:
  - Accuracy
  - Top-1, Top-5
  - Others?